In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
train=pd.read_csv("/kaggle/input/titanic/train.csv")
test=pd.read_csv("/kaggle/input/titanic/test.csv")

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
train.describe()

In [ ]:
train.info()

In [ ]:
train.isnull().sum()

In [ ]:
train.duplicated().sum()

**Exploratory Data Analysis**

***Age and Sex Based Analysis***


In [ ]:
sns.countplot(x=train['Sex'])

More number of males than females on the ship.

In [ ]:
sns.barplot(x=train['Survived'],y=train['Age'],hue=train['Sex'])

Replacing the null values in 'Age' column with mean:

In [ ]:
train['Age'].fillna(train['Age'].mean(), inplace=True)

In [ ]:
test['Age'].fillna(test['Age'].mean(), inplace=True)

***PClass Based Analysis***

In [ ]:
sns.countplot(x=train['Pclass'])

The above figure shows that there were highest number of passengers travelling in Pclass=3, followed by Pclass=1 and then Pclass=2. To find the reason behind this, next we can evaluate the fares of all three classes.

In [ ]:
sns.barplot(x=train['Pclass'],y=train['Fare'])

In [ ]:
print(train['Fare'][train['Pclass']==1].mean())
print(train['Fare'][train['Pclass']==2].mean())
print(train['Fare'][train['Pclass']==3].mean())


* The Pclass 3 tickets had the lowest fares, hence most passengers travelled in it.
* The mean fare of Pclass 1 was very high, which justifies fewer wealthier passengers travelling by it.
* Pclass 2, however had even lesser passesngers than Pclass 1, despite having significantly lesser mean fare. A possible reason for this could be because there was not a significant difference in fare between Pclass 2 and Pclass 3, leading most budget conscious passengers to choose the cheaper Pclass 3.







In [ ]:
sns.barplot(x=train['Pclass'],y=train['Survived'])

In [ ]:
pd.crosstab(train['Pclass'],train['Survived'])

In [ ]:
train.groupby('Pclass')['Survived'].mean()*100

The survival rate was highest in Pclass 1, followed by Pclass 2 and then the least in Pclass 3(which had the most number of passengers).

***SibSp and Parch Analysis***

In [ ]:
train['Family']=train['SibSp']+train['Parch']
test['Family']=test['SibSp']+test['Parch']

In [ ]:
train.head()

A new feature 'Family' is created by combining the SibSp (siblings/spouses aboard) and Parch (parents/children aboard) columns to denote the total number of family members aboard. Consequently, after creating this feature, the SipSb and Parch are then dropped from the dataset.

In [ ]:
train.drop(columns=['SibSp','Parch'], inplace=True)
test.drop(columns=['SibSp','Parch'], inplace=True)

In [ ]:
train.head()

***Cabin Based Analysis***

In [ ]:
train['HasCabin']=train['Cabin'].notnull().astype(int)
test['HasCabin']=test['Cabin'].notnull().astype(int)

In [ ]:
train.drop(columns=['Cabin'],inplace=True)
train.head()


In [ ]:
test.drop(columns=['Cabin'],inplace=True)
test.head()

In [ ]:
sns.barplot(x=train['Pclass'],y=train['HasCabin'], hue=train['Survived'])

Almost all Pclass 1 passengers had cabins. Very few to no cabins for Pclass 2 and Pclass 3 passengers.

In [ ]:
sns.barplot(x=train['HasCabin'],y=train['Survived'])

The above plot shows that having a cabin showed increased chances of survival. This is also evident by the fact that only first class passengers had cabins, which indeed had higher survival than Pclass 2 and 3.

***Other Analysis***

In [ ]:
sns.countplot(x=train['Embarked'])

Most passengers boarded at Southampton as compared to Cherbourg and Queenstown.

In [ ]:
train.info()

In [ ]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()

In [ ]:
train['Sex']=le.fit_transform(train['Sex']) #male=1,female=0
train.head()

In [ ]:
test['Sex']=le.fit_transform(test['Sex']) #male=1,female=0

In [ ]:
train['Embarked']=le.fit_transform(train['Embarked']) #0,1,2
train.head()

In [ ]:
test['Embarked']=le.fit_transform(test['Embarked']) #0,1,2

In [ ]:
train['Embarked'].fillna(train['Embarked'].mean(), inplace=True)

In [ ]:
train.info()

In [ ]:
test.info()

In [ ]:
test['Fare'].fillna(train['Fare'].mean(), inplace=True)

**Model Evaluation and Prediction**

In [ ]:
X_train=train[['Pclass','Sex','Age','Fare','Embarked','Family','HasCabin']]
y_train=train['Survived']

In [ ]:
X_test=test[['Pclass','Sex','Age','Fare','Embarked','Family','HasCabin']]

In [ ]:
print(X_train.shape)
print(y_train.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [ ]:
models = { "Logistic Regression": LogisticRegression(max_iter=1000),
           "Naive Bayes": GaussianNB(),
           "Decision Tree": DecisionTreeClassifier(random_state=42),
           "Random Forest": RandomForestClassifier(random_state=42),
           "Perceptron": Perceptron(max_iter=1000,random_state=42),
           "XGBoost": XGBClassifier(random_state=42)}

In [ ]:
for name,model in models.items():
  model.fit(X_train,y_train)
  y_pred=model.predict(X_test)
  a=model.score(X_train,y_train)*100
  print(f"{name}: {a}")

In [ ]:
rf=RandomForestClassifier(random_state=42)
rf.fit(X_train,y_train)
y_pred=rf.predict(X_test)

In [ ]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": y_pred
})

In [ ]:
submission.to_csv("titanic.csv", index=False)